# Model Evaluation & Error Analysis - Workout Monitoring System

## Purpose

**WHY comprehensive evaluation?**

1. **Beyond accuracy**: Single metrics hide important details
2. **Per-class analysis**: Some exercises may be harder to classify
3. **Error analysis**: Understand WHY the model fails
4. **Threshold tuning**: Optimize decision boundaries
5. **Confidence calibration**: Are probability estimates reliable?

### What We'll Analyze:
- Confusion matrices (per-class performance)
- Precision/Recall trade-offs
- ROC curves and AUC
- Error case analysis
- Feature contribution to errors

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✓ Libraries imported")

## 1. Load Model and Data

We'll load the best model saved from the previous notebook.

In [ ]:
# Load saved model and scaler
model = joblib.load('../models/best_model.pkl')
scaler = joblib.load('../models/scaler.pkl')

# Load test data
test_data = np.load('../data/balanced/test.npz')
X_test = test_data['X']
y_test = test_data['y']

# Scale test data
X_test_scaled = scaler.transform(X_test)

# Get predictions
y_pred = model.predict(X_test_scaled)

# Get probability predictions if available
if hasattr(model, 'predict_proba'):
    y_prob = model.predict_proba(X_test_scaled)
    has_proba = True
else:
    has_proba = False

print(f"✓ Loaded model: {type(model).__name__}")
print(f"✓ Test samples: {len(y_test)}")
print(f"✓ Classes: {np.unique(y_test)}")

## 2. Classification Report

**WHY use classification report?**

- Shows **per-class** metrics (not just overall)
- Reveals if model struggles with specific classes
- Support column shows class sizes

### Metrics Explained:

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **Precision** | TP / (TP + FP) | "When I predict X, am I right?" |
| **Recall** | TP / (TP + FN) | "Do I find all X's?" |
| **F1** | 2 × (P × R) / (P + R) | Balance of precision and recall |
| **Support** | Count | Number of samples in class |

In [ ]:
# Print classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred, zero_division=0))

# Overall metrics
print("\nOverall Metrics:")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print(f"  F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")

## 3. Confusion Matrix Analysis

**WHY analyze confusion matrix?**

1. **See misclassification patterns**: Which classes get confused?
2. **Identify systematic errors**: Consistent mistakes reveal model limitations
3. **Guide improvements**: Know where to focus efforts

### How to Read:
- **Diagonal**: Correct predictions (higher = better)
- **Off-diagonal**: Errors (row = actual, column = predicted)
- **Row sum**: Total samples of that class
- **Column sum**: Total predictions of that class

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Normalize for percentage view
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Class')
axes[0].set_ylabel('Actual Class')

# Normalized (percentages)
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Greens', ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Class')
axes[1].set_ylabel('Actual Class')

plt.tight_layout()
plt.show()

# Analyze errors
print("\n📊 Confusion Matrix Insights:")
total = cm.sum()
correct = np.trace(cm)
print(f"  Correct predictions: {correct}/{total} ({correct/total*100:.1f}%)")
print(f"  Errors: {total - correct}/{total} ({(total-correct)/total*100:.1f}%)")

## 4. Per-Class Performance

**WHY analyze per-class?**

- Overall metrics can mask poor performance on minority classes
- Some exercises may be inherently harder to distinguish
- Guides data collection: need more examples of hard classes?

In [ ]:
# Per-class metrics
classes = np.unique(y_test)

class_metrics = []
for c in classes:
    # Binary for this class
    y_true_binary = (y_test == c).astype(int)
    y_pred_binary = (y_pred == c).astype(int)
    
    class_metrics.append({
        'Class': c,
        'Samples': (y_test == c).sum(),
        'Precision': precision_score(y_true_binary, y_pred_binary, zero_division=0),
        'Recall': recall_score(y_true_binary, y_pred_binary, zero_division=0),
        'F1-Score': f1_score(y_true_binary, y_pred_binary, zero_division=0)
    })

class_df = pd.DataFrame(class_metrics)
print("Per-Class Performance:")
print("=" * 60)
print(class_df.to_string(index=False))

# Identify best and worst classes
best_class = class_df.loc[class_df['F1-Score'].idxmax()]
worst_class = class_df.loc[class_df['F1-Score'].idxmin()]

print(f"\n🏆 Best performing class: {best_class['Class']} (F1: {best_class['F1-Score']:.3f})")
print(f"⚠️  Worst performing class: {worst_class['Class']} (F1: {worst_class['F1-Score']:.3f})")

In [ ]:
# Visualize per-class performance
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(classes))
width = 0.25

ax.bar(x - width, class_df['Precision'], width, label='Precision', color='steelblue', alpha=0.8)
ax.bar(x, class_df['Recall'], width, label='Recall', color='coral', alpha=0.8)
ax.bar(x + width, class_df['F1-Score'], width, label='F1-Score', color='green', alpha=0.8)

ax.set_xlabel('Class')
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 5. ROC Curves and AUC

**WHY use ROC curves?**

1. **Threshold-independent**: Shows performance across all decision thresholds
2. **Visual comparison**: Easy to compare multiple classifiers
3. **AUC summarizes**: Single number capturing overall discrimination ability

### How to Read:
- **Diagonal line**: Random classifier (AUC = 0.5)
- **Top-left corner**: Perfect classifier (AUC = 1.0)
- **Curve closer to top-left**: Better performance

### AUC Interpretation:
- **0.9-1.0**: Excellent
- **0.8-0.9**: Good
- **0.7-0.8**: Fair
- **< 0.7**: Poor

In [ ]:
if has_proba:
    # Binarize the output for multi-class ROC
    n_classes = len(np.unique(y_test))
    y_test_bin = label_binarize(y_test, classes=np.unique(y_test))
    
    # Handle binary case
    if n_classes == 2:
        y_test_bin = np.hstack([1 - y_test_bin, y_test_bin])
    
    # Compute ROC curve for each class
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
    
    for i, color in zip(range(n_classes), colors):
        if y_test_bin.shape[1] > i:
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, lw=2,
                   label=f'Class {i} (AUC = {roc_auc:.3f})')
    
    # Plot diagonal
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.5)')
    
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Model doesn't support probability predictions. ROC curves unavailable.")

## 6. Error Analysis

**WHY analyze errors?**

1. **Understand failure modes**: Why does the model make mistakes?
2. **Identify data issues**: Bad labels? Noisy data?
3. **Guide improvements**: More data? Better features? Different model?
4. **Set expectations**: Know model limitations

In [ ]:
# Find misclassified samples
errors = y_test != y_pred
error_indices = np.where(errors)[0]

print(f"Error Analysis:")
print("=" * 60)
print(f"Total samples: {len(y_test)}")
print(f"Correct: {(~errors).sum()} ({(~errors).sum()/len(y_test)*100:.1f}%)")
print(f"Errors: {errors.sum()} ({errors.sum()/len(y_test)*100:.1f}%)")

if len(error_indices) > 0:
    print(f"\nError samples (first 10):")
    print(f"{'Index':<10} {'Actual':<10} {'Predicted':<10}")
    print("-" * 30)
    for idx in error_indices[:10]:
        print(f"{idx:<10} {y_test[idx]:<10} {y_pred[idx]:<10}")
else:
    print("\n✓ No errors! Perfect classification.")

In [ ]:
# Analyze error distribution
if len(error_indices) > 0:
    error_actual = y_test[errors]
    error_pred = y_pred[errors]
    
    # Which classes are most often misclassified?
    actual_counts = pd.Series(error_actual).value_counts()
    pred_counts = pd.Series(error_pred).value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    actual_counts.plot(kind='bar', ax=axes[0], color='coral', alpha=0.8)
    axes[0].set_title('Errors by Actual Class', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Actual Class')
    axes[0].set_ylabel('Error Count')
    
    pred_counts.plot(kind='bar', ax=axes[1], color='steelblue', alpha=0.8)
    axes[1].set_title('Errors by Predicted Class', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Predicted Class')
    axes[1].set_ylabel('Error Count')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Error Insights:")
    print(f"  Most often misclassified actual class: {actual_counts.idxmax()}")
    print(f"  Most often incorrectly predicted: {pred_counts.idxmax()}")

## 7. Summary & Recommendations

### Model Performance Summary

Based on our comprehensive evaluation:

| Aspect | Finding | Implication |
|--------|---------|-------------|
| **Overall Accuracy** | See results above | General correctness |
| **Per-Class** | Varies by class | May need more data for weak classes |
| **Confusion Patterns** | See matrix | Reveals systematic confusions |
| **Error Cases** | Analyzed above | Guide for improvements |

### Recommendations

1. **If accuracy is low on specific classes**: Collect more data for those classes
2. **If precision < recall**: Model is over-predicting that class
3. **If recall < precision**: Model is missing true positives
4. **If classes are confused**: Features may not distinguish them well

### Next Steps

1. **Deploy model** for real-time inference
2. **Collect user feedback** to identify edge cases
3. **Retrain periodically** with new data
4. **Consider ensemble** of models for robustness

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("MODEL EVALUATION COMPLETE!")
print("=" * 70)

print("\n📊 Final Results:")
print(f"  Model: {type(model).__name__}")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"  F1-Score: {f1_score(y_test, y_pred, average='weighted', zero_division=0):.3f}")
print(f"  Error Rate: {(y_test != y_pred).sum() / len(y_test) * 100:.1f}%")

print("\n✅ Evaluation complete!")
print("📁 Model saved to: models/best_model.pkl")
print("\n🚀 Ready for deployment!")